# Análisis OLAP Interactivos con Plotly
Este Notebook se conecta a la base de datos `invernadero_olap_fase2` para ejecutar las consultas analíticas de la **Fase 2** del proyecto y visualizarlas interactivamente.

In [1]:
import mysql.connector
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import getpass
import warnings

# Suprimir warnings de pandas sobre SQLAlchemy
warnings.filterwarnings('ignore', category=UserWarning)

print("Por favor, ingresa tu contraseña de MySQL para root:")
password = getpass.getpass()

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=password,
    database="invernadero_olap_fase2",
    port=3306
)
print("¡Conectado a la base de datos OLAP!")

Por favor, ingresa tu contraseña de MySQL para root:
¡Conectado a la base de datos OLAP!


## 1. Anomalías por Sensor
¿Qué sensores presentan más registros fuera de sus umbrales óptimos?

In [3]:
query_anomalias = """
SELECT 
    s.nombre_sensor,
    s.variable_medida,
    s.ubicacion,
    COUNT(CASE WHEN f.flag_anomalia = 1 THEN 1 END) AS total_anomalias,
    COUNT(*) AS total_mediciones,
    ROUND((COUNT(CASE WHEN f.flag_anomalia = 1 THEN 1 END) / COUNT(*)) * 100, 2) AS porcentaje_anomalo
FROM fact_mediciones_invernadero f
JOIN dim_sensor s ON f.id_sensor = s.id_sensor
GROUP BY s.nombre_sensor, s.variable_medida, s.ubicacion
ORDER BY total_anomalias DESC;
"""

df_anomalias = pd.read_sql(query_anomalias, conn)

fig = px.bar(
    df_anomalias, 
    x='total_anomalias', 
    y='nombre_sensor', 
    color='porcentaje_anomalo',
    orientation='h',
    text='total_anomalias',
    title="<b>Total de Anomalías Detectadas por Sensor</b>",
    color_continuous_scale="Reds",
    labels={'total_anomalias': 'Número de Anomalías', 'nombre_sensor': 'Sensor', 'porcentaje_anomalo': '% Anómalo'}
)

fig.update_traces(textposition='outside', marker_line_color='rgb(8,48,107)', marker_line_width=1.5, opacity=0.9)
fig.update_layout(
    template="plotly_dark",
    yaxis={'categoryorder':'total ascending'},
    font=dict(family="Inter, Arial, sans-serif", size=14, color="white"),
    margin=dict(l=10, r=20, t=80, b=50),
    plot_bgcolor='#0E1117',
    paper_bgcolor='#0E1117'
)
fig.update_xaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)', title="")
fig.update_yaxes(showgrid=False, title="")
fig.show()

## 2. Comportamiento Térmico por Franja Horaria
¿Cómo varía la temperatura entre el interior y el exterior según el momento del día?

In [29]:
query_temp = """
SELECT 
    s.nombre_sensor,
    t.franja_horaria,
    ROUND(AVG(f.valor_medido), 2) AS temperatura_promedio
FROM fact_mediciones_invernadero f
JOIN dim_sensor s ON f.id_sensor = s.id_sensor
JOIN dim_tiempo t ON f.id_tiempo = t.id_tiempo
WHERE s.variable_medida = 'Temperatura Ambiental'
GROUP BY s.nombre_sensor, t.franja_horaria
ORDER BY s.nombre_sensor, FIELD(t.franja_horaria, 'Madrugada', 'Mañana', 'Tarde', 'Noche');
"""

df_temp = pd.read_sql(query_temp, conn)

fig = px.line(
    df_temp, 
    x='franja_horaria', 
    y='temperatura_promedio', 
    color='nombre_sensor',
    markers=True,
    text='temperatura_promedio',
    title="<b>Fluctuación Térmica Diaria (Franja Horaria)</b>",
    color_discrete_sequence=['#00CC96', '#EF553B']
)

fig.update_traces(textposition="top center", line=dict(width=4), marker=dict(size=12, line=dict(width=2, color='white')))
fig.update_layout(
    template="plotly_dark",
    font=dict(family="Inter, Arial, sans-serif", size=14, color="white"),
    hovermode="x unified",
    margin=dict(l=10, r=20, t=80, b=50),
    plot_bgcolor='#0E1117',
    paper_bgcolor='#0E1117',
    legend=dict(title=None, orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.update_yaxes(gridcolor='rgba(255, 255, 255, 0.1)', title="")
fig.update_xaxes(gridcolor='rgba(255, 255, 255, 0.1)', title="")
fig.show()

## 3. Demanda de Riego por Tratamiento de Suelo
¿Cuántas horas requirió riego cada surco según la época del año?

In [30]:
query_riego = """
SELECT 
    su.id_surco,
    su.nombre_surco,
    su.humedad_objetivo_pct,
    t.nombre_mes,
    t.mes,
    SUM(f.flag_riego) AS horas_riego_requerido
FROM fact_mediciones_invernadero f
JOIN dim_surco_suelo su ON f.id_surco = su.id_surco
JOIN dim_tiempo t ON f.id_tiempo = t.id_tiempo
WHERE su.id_surco IN (1, 2, 3)
GROUP BY su.id_surco, su.nombre_surco, su.humedad_objetivo_pct, t.nombre_mes, t.mes
ORDER BY su.id_surco, t.mes;
"""

df_riego = pd.read_sql(query_riego, conn)

fig = px.area(
    df_riego, 
    x='nombre_mes', 
    y='horas_riego_requerido', 
    color='nombre_surco',
    title="<b>Evolución del Déficit Hídrico Mensual</b>",
    color_discrete_sequence=['#636EFA', '#00CC96', '#AB63FA']
)

fig.update_traces(mode='lines+markers', marker=dict(size=8, line=dict(width=1, color='white')), fill='tonexty')
fig.update_layout(
    template="plotly_dark",
    font=dict(family="Inter, Arial, sans-serif", size=14, color="white"),
    hovermode="x unified",
    margin=dict(l=10, r=20, t=80, b=50),
    plot_bgcolor='#0E1117',
    paper_bgcolor='#0E1117',
    legend=dict(title=None, orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.update_yaxes(showgrid=True, gridcolor='rgba(255, 255, 255, 0.1)', title="")
fig.update_xaxes(showgrid=True, gridcolor='rgba(255, 255, 255, 0.1)', title="")
fig.show()

## 4. Evolución Temporal: Temperatura Diaria (Interior vs Exterior)

In [31]:
query_ts_temp = """
SELECT 
    t.fecha,
    s.nombre_sensor,
    ROUND(AVG(f.valor_medido), 2) AS temp_promedio
FROM fact_mediciones_invernadero f
JOIN dim_sensor s ON f.id_sensor = s.id_sensor
JOIN dim_tiempo t ON f.id_tiempo = t.id_tiempo
WHERE s.variable_medida = 'Temperatura Ambiental'
GROUP BY t.fecha, s.nombre_sensor
ORDER BY t.fecha;
"""
df_ts_temp = pd.read_sql(query_ts_temp, conn)

fig = px.line(df_ts_temp, x='fecha', y='temp_promedio', color='nombre_sensor',
              title="<b>Temperatura promedio diaria (°C)</b>",
              color_discrete_sequence=['#00CC96', '#636EFA'])
fig.update_layout(template="plotly_dark", plot_bgcolor='#0E1117', paper_bgcolor='#0E1117',
                  legend=dict(title=None, orientation="h", y=1.02, x=0))
fig.update_xaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)', title="")
fig.update_yaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)', title="")
fig.show()

## 5. Evolución Temporal: Humedad Relativa Diaria

In [32]:
query_ts_hum = """
SELECT 
    t.fecha,
    s.nombre_sensor,
    ROUND(AVG(f.valor_medido), 2) AS hum_promedio
FROM fact_mediciones_invernadero f
JOIN dim_sensor s ON f.id_sensor = s.id_sensor
JOIN dim_tiempo t ON f.id_tiempo = t.id_tiempo
WHERE s.variable_medida = 'Humedad Relativa'
GROUP BY t.fecha, s.nombre_sensor
ORDER BY t.fecha;
"""
df_ts_hum = pd.read_sql(query_ts_hum, conn)

fig = px.line(df_ts_hum, x='fecha', y='hum_promedio', color='nombre_sensor', line_dash='nombre_sensor',
              title="<b>Humedad Relativa (%)</b>",
              color_discrete_sequence=['#AB63FA', '#FFA15A'])
fig.update_layout(template="plotly_dark", plot_bgcolor='#0E1117', paper_bgcolor='#0E1117',
                  legend=dict(title=None, orientation="h", y=1.02, x=0))
fig.update_xaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)', title="")
fig.update_yaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)', title="")
fig.show()

## 6. Patrón de Luminosidad Promedio por Hora del Día

In [2]:
query_lux = """
SELECT 
    t.hora,
    s.nombre_sensor,
    ROUND(AVG(f.valor_medido), 2) AS lux_promedio
FROM fact_mediciones_invernadero f
JOIN dim_sensor s ON f.id_sensor = s.id_sensor
JOIN dim_tiempo t ON f.id_tiempo = t.id_tiempo
WHERE s.variable_medida = 'Luminosidad'
GROUP BY t.hora, s.nombre_sensor
ORDER BY t.hora;
"""
df_lux = pd.read_sql(query_lux, conn)

fig = px.bar(df_lux, x='hora', y='lux_promedio', color='nombre_sensor', barmode='group',
             title="<b>Lux promedio por hora</b>",
             color_discrete_sequence=['#FFB703', "#1DE82E"])
fig.update_layout(template="plotly_dark", plot_bgcolor='#0E1117', paper_bgcolor='#0E1117',
                  legend=dict(title=None, orientation="h", y=1.02, x=0))
fig.update_xaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)', title="", dtick=5)
fig.update_yaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)', title="")
fig.show()

## 7. Humedad de Suelo a Distintas Profundidades (Tratamientos)

In [34]:
query_suelo = """
SELECT 
    t.fecha_hora,
    s.nombre_sensor,
    f.valor_medido
FROM fact_mediciones_invernadero f
JOIN dim_sensor s ON f.id_sensor = s.id_sensor
JOIN dim_tiempo t ON f.id_tiempo = t.id_tiempo
WHERE s.nombre_sensor LIKE 'LM393%'
ORDER BY t.fecha_hora;
"""
df_suelo = pd.read_sql(query_suelo, conn)

fig = px.line(df_suelo, x='fecha_hora', y='valor_medido', color='nombre_sensor',
              title="<b>Humedad de suelo a distintas profundidades (%)</b>",
              color_discrete_sequence=['#00CC96', '#3366FF', '#AB63FA'])
fig.update_traces(line=dict(width=1))
fig.update_layout(template="plotly_dark", plot_bgcolor='#0E1117', paper_bgcolor='#0E1117',
                  legend=dict(title=None, orientation="h", y=1.02, x=0))
fig.update_xaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)', title="")
fig.update_yaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)', title="")
fig.show()

## 8. Mapa de Calor: Patrón de Temperatura Interior (Hora x Mes)

In [35]:
query_hm = """
SELECT 
    t.mes,
    t.nombre_mes,
    t.hora,
    ROUND(AVG(f.valor_medido), 2) AS temp_promedio
FROM fact_mediciones_invernadero f
JOIN dim_sensor s ON f.id_sensor = s.id_sensor
JOIN dim_tiempo t ON f.id_tiempo = t.id_tiempo
WHERE s.nombre_sensor = 'DHT20 Interior' AND s.variable_medida = 'Temperatura Ambiental'
GROUP BY t.mes, t.nombre_mes, t.hora
ORDER BY t.mes DESC, t.hora;
"""
df_hm = pd.read_sql(query_hm, conn)
heatmap_data = df_hm.pivot(index="nombre_mes", columns="hora", values="temp_promedio")
# Reordenar los meses cronológicamente inverso
meses = ['Diciembre', 'Noviembre', 'Octubre', 'Septiembre', 'Agosto', 'Julio', 'Junio', 'Mayo', 'Abril', 'Marzo', 'Febrero', 'Enero']
heatmap_data = heatmap_data.reindex(meses)

fig = px.imshow(heatmap_data, 
                labels=dict(x="Hora del día", y="", color="°C"),
                x=[f"{h:02d}h" for h in range(24)],
                title="<b>Temperatura interior promedio (°C) por hora y mes</b>",
                color_continuous_scale="Turbo")
fig.update_layout(template="plotly_dark", plot_bgcolor='#0E1117', paper_bgcolor='#0E1117')
fig.show()

## 9. Distribución de Lecturas por Franja Horaria

In [36]:
query_franja = """
SELECT 
    franja_horaria,
    COUNT(DISTINCT id_tiempo) AS total_horas
FROM dim_tiempo
GROUP BY franja_horaria;
"""
df_franja = pd.read_sql(query_franja, conn)

fig = px.pie(df_franja, values='total_horas', names='franja_horaria', hole=0.5,
             title="<b>Distribución de horas registradas por franja</b>",
             color_discrete_sequence=['#00CC96', '#AB63FA', '#FF8C00', '#0099FF'])
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(template="plotly_dark", plot_bgcolor='#0E1117', paper_bgcolor='#0E1117', showlegend=False)
fig.show()

## 10. Histograma de Frecuencias: Temperatura Interior

In [37]:
query_hist = """
SELECT valor_medido
FROM fact_mediciones_invernadero f
JOIN dim_sensor s ON f.id_sensor = s.id_sensor
WHERE s.nombre_sensor = 'DHT20 Interior' AND s.variable_medida = 'Temperatura Ambiental';
"""
df_hist = pd.read_sql(query_hist, conn)
media = df_hist['valor_medido'].mean()
umbral_alerta = 38.0 # Según dim_sensor

fig = px.histogram(df_hist, x="valor_medido", nbins=40,
                   title="<b>Frecuencia de temperatura (°C)</b>",
                   color_discrete_sequence=['#2CA02C'])
                   
# Línea de la media
fig.add_vline(x=media, line_width=2, line_dash="solid", line_color="#0099FF", 
              annotation_text=f"Media {media:.1f}°C", annotation_position="top right")
# Línea de alerta
fig.add_vline(x=umbral_alerta, line_width=2, line_dash="dash", line_color="red", 
              annotation_text=f"Umbral {umbral_alerta}°C", annotation_position="top left")

fig.update_layout(template="plotly_dark", plot_bgcolor='#0E1117', paper_bgcolor='#0E1117',
                  xaxis_title="", yaxis_title="")
fig.update_xaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)')
fig.update_yaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)')
fig.show()

## 11. Correlación: Temperatura vs Humedad (Interior)

In [38]:
query_corr = """
SELECT 
    t.id_tiempo,
    MAX(CASE WHEN s.variable_medida = 'Temperatura Ambiental' THEN f.valor_medido END) AS temp,
    MAX(CASE WHEN s.variable_medida = 'Humedad Relativa' THEN f.valor_medido END) AS hum,
    t.franja_horaria
FROM fact_mediciones_invernadero f
JOIN dim_sensor s ON f.id_sensor = s.id_sensor
JOIN dim_tiempo t ON f.id_tiempo = t.id_tiempo
WHERE s.nombre_sensor = 'DHT20 Interior'
GROUP BY t.id_tiempo, t.franja_horaria;
"""
df_corr = pd.read_sql(query_corr, conn)
df_corr = df_corr.dropna()

fig = px.scatter(df_corr, x="temp", y="hum", color="franja_horaria", opacity=0.6,
                 title="<b>Correlación Térmica/Hídrica Interior</b>",
                 labels={"temp": "Temperatura (°C)", "hum": "Humedad Relativa (%)"},
                 color_discrete_sequence=['#00CC96', '#AB63FA', '#FF8C00', '#0099FF'])

fig.update_layout(template="plotly_dark", plot_bgcolor='#0E1117', paper_bgcolor='#0E1117')
fig.update_xaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)')
fig.update_yaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)')
fig.show()

## 12. Comportamiento Promedio por Día de la Semana

In [39]:
query_dia = """
SELECT 
    t.dia_semana,
    ROUND(AVG(CASE WHEN s.variable_medida = 'Temperatura Ambiental' THEN f.valor_medido END), 2) AS temp_avg,
    ROUND(AVG(CASE WHEN s.variable_medida = 'Humedad Relativa' THEN f.valor_medido END), 2) AS hum_avg
FROM fact_mediciones_invernadero f
JOIN dim_sensor s ON f.id_sensor = s.id_sensor
JOIN dim_tiempo t ON f.id_tiempo = t.id_tiempo
WHERE s.ubicacion = 'Interior'
GROUP BY t.dia_semana
ORDER BY FIELD(t.dia_semana, 'Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo');
"""
df_dia = pd.read_sql(query_dia, conn)

import plotly.graph_objects as go

fig = go.Figure(data=[
    go.Bar(name='Temperatura Promedio (°C)', x=df_dia['dia_semana'], y=df_dia['temp_avg'], marker_color='#EF553B'),
    go.Bar(name='Humedad Promedio (%)', x=df_dia['dia_semana'], y=df_dia['hum_avg'], marker_color='#636EFA')
])
fig.update_layout(title="<b>Promedios Semanales (Interior)</b>", barmode='group',
                  template="plotly_dark", plot_bgcolor='#0E1117', paper_bgcolor='#0E1117',
                  legend=dict(orientation="h", y=1.02, x=0))
fig.update_xaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)')
fig.update_yaxes(showgrid=True, gridcolor='rgba(255,255,255,0.1)')
fig.show()